<a href="https://colab.research.google.com/github/JonasCandid0/TCC-Univesp/blob/main/tcc_analise_espacial_simba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bibliotecas

In [ ]:
!pip install geopandas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import seaborn as sns
from shapely.geometry import Point, LineString
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde
from mpl_toolkits.axes_grid1 import make_axes_locatable



# Tratemento inicial
---

### Esforços de Monitoramento

In [ ]:
caminho_monitoramento ='/content/drive/MyDrive/fontes de dados SIMBA/Esforcos_de_Monitoramento.csv'

In [ ]:
colunas_esforco = [
    'Estado', 'Cidade', 'Praia', 'Trecho',
    'Tipo', 'Estratégia', 'Tipo de veículo',
    'Data/Hora início', 'Data/Hora término',
    'Latitude ponto inicial', 'Longitude ponto inicial',
    'Latitude ponto final', 'Longitude ponto final',
]

In [ ]:
esforco_monitora_filtrada = pd.read_csv(
    caminho_monitoramento,
    sep=";",
    usecols=colunas_esforco ,
    low_memory=False
)

In [ ]:
esforco_monitora_filtrada.head()

In [ ]:
esforco_monitora_filtrada.info()

In [ ]:
# Conversão de tipos — Esforços de Monitoramento

# 1. Datas
esforco_monitora_filtrada['Data/Hora início'] = pd.to_datetime(
    esforco_monitora_filtrada['Data/Hora início'], dayfirst=True, errors='coerce'
)
esforco_monitora_filtrada['Data/Hora término'] = pd.to_datetime(
    esforco_monitora_filtrada['Data/Hora término'], dayfirst=True, errors='coerce'
)

In [ ]:
# 2. Variáveis categóricas (baixa cardinalidade — economiza memória RAM)
colunas_cat_esforco = ['Estado', 'Cidade', 'Praia', 'Trecho', 'Tipo', 'Estratégia', 'Tipo de veículo']
for col in colunas_cat_esforco:
    esforco_monitora_filtrada[col] = esforco_monitora_filtrada[col].astype('category')

print(' Esforços — tipos convertidos:')
esforco_monitora_filtrada.dtypes

In [ ]:
esforco_monitora_filtrada.head()

In [ ]:
esforco_monitora_filtrada.head()

### Biometria

In [ ]:
caminho_biometria ='/content/drive/MyDrive/fontes de dados SIMBA/Biometrias_arrumando.csv'

In [ ]:
biometria = pd.read_csv(caminho_biometria, sep=";", encoding='utf-8')

In [ ]:
biometria.info()

In [ ]:
colunas_selecionadas = ["Identificador da ocorrência", "Identificador do indivíduo", "Espécies - Classe",
                         "Espécies - Espécie", "Ficha de campo", "Instituição executora", "Tipo", "Responsável"
]

biometria_filtrada = biometria[colunas_selecionadas]

In [ ]:
biometria_filtrada.head()

In [ ]:
# Conversão de tipos — Biometria

# 1. Desvinculamos a tabela da visualização original
biometria_filtrada = biometria_filtrada.copy()

# Variáveis categóricas (taxonomia e classificações repetitivas)
colunas_cat_biometria = [
    'Espécies - Classe', 'Espécies - Espécie',
    'Instituição executora', 'Tipo', 'Responsável'
]

for col in colunas_cat_biometria:
    if col in biometria_filtrada.columns:
        biometria_filtrada[col] = biometria_filtrada[col].astype('category')

# Identificadores únicos permanecem como object (alta cardinalidade — não se beneficiam de category)
# 'Identificador da ocorrência', 'Identificador do indivíduo', 'Ficha de campo'

print(' Biometria — tipos convertidos com sucesso e sem warnings:')
display(biometria_filtrada.dtypes)

In [ ]:
biometria_filtrada.head()

### Ocorrencias de fauna alvo

In [ ]:
caminho_arquivo = '/content/drive/MyDrive/fontes de dados SIMBA/O_fauna_alvo_individual.csv'

In [ ]:
colunas_selecionadas = [
    'Identificador do indivíduo', 'Identificador da ocorrência',
    'Data/Hora', 'Estado', 'Cidade', 'Praia',
    'Ponto - Lat', 'Ponto - Long',
    'Espécies - Classe', 'Espécies - Espécie',
    'Estágio de desenvolvimento', 'Espécie ameaçada',
    'Condição', 'Condição da carcaça',
    'Presença de óleo', 'Interações antrópicas', 'Causa da morte'
]

In [ ]:
ocorrencias_filtradas = pd.read_csv(
        caminho_arquivo,
        sep=';',
        encoding='utf-8',
        usecols=colunas_selecionadas,
        low_memory=False
    )

In [ ]:
ocorrencias_filtradas.info()

In [ ]:
ocorrencias_filtradas.head()

In [ ]:
# 1. Data/Hora
ocorrencias_filtradas['Data/Hora'] = pd.to_datetime(
    ocorrencias_filtradas['Data/Hora'], dayfirst=True, errors='coerce'
)

In [ ]:
colunas_cat_ocorrencias = [
    'Estado', 'Cidade', 'Praia',
    'Espécies - Classe', 'Espécies - Espécie',
    'Estágio de desenvolvimento', 'Espécie ameaçada',
    'Condição', 'Presença de óleo', 'Causa da morte'
]

for col in colunas_cat_ocorrencias:
    if col in ocorrencias_filtradas.columns:
        ocorrencias_filtradas[col] = ocorrencias_filtradas[col].astype('category')

In [ ]:
ocorrencias_filtradas.info()



---


## Relacionamento e Higienização

In [ ]:
# Merge principal: Ocorrências + Biometria
# how='left' garante que todos os encalhes são mantidos,
# mesmo os que não têm biometria registrada.

linhas_antes = ocorrencias_filtradas.shape[0]

df_master = pd.merge(
    ocorrencias_filtradas,
    biometria_filtrada,
    on=['Identificador da ocorrência', 'Identificador do indivíduo'],
    how='left',
    suffixes=('_ocor', '_bio')
)

# Remove duplicatas que podem surgir quando um indivíduo
# tem mais de um registro na tabela de Biometria
df_master = df_master.drop_duplicates(
    subset=['Identificador da ocorrência', 'Identificador do indivíduo']
)

linhas_depois = df_master.shape[0]

print(f' Relatório do Merge:')
print(f'   Linhas em Ocorrências (antes): {linhas_antes}')
print(f'   Linhas em df_master (depois):  {linhas_depois}')

if linhas_antes == linhas_depois:
    print(' Merge perfeito! Nenhuma ocorrência foi perdida ou duplicada.')
else:
    print('  ATENÇÃO: O número de linhas mudou. Verifique a cardinalidade dos dados.')


In [ ]:
# Visualização das primeiras linhas do dataset master
print(f'Colunas do df_master: {list(df_master.columns)}')
print(f'Shape: {df_master.shape}')
display(df_master.head())


In [ ]:
print(df_master.shape)
print(df_master.isnull().sum()[df_master.isnull().sum() > 0])


In [ ]:
pd.set_option('display.max_columns', None)
display(df_master.head())

> **Nota sobre Esforços de Monitoramento:**  
> O dataset `esforco_monitora_filtrada` não participa do merge principal porque não possui
> as colunas `Identificador da ocorrência` / `Identificador do indivíduo`.  
> Ele será usado nas próximas etapas separadamente, para normalizar a taxa de encalhes
> pelo esforço de monitoramento de cada praia e período.




---


## Análise Exploratória e Séries Temporais

1.1 Preparação do índice temporal

In [ ]:
# Define Data/Hora como índice do df_master para uso com .resample()
df_temporal = df_master.set_index('Data/Hora').sort_index()

print(f'Período coberto: {df_temporal.index.min().date()} → {df_temporal.index.max().date()}')
print(f'Total de registros: {len(df_temporal)}')


1.2 Evolução temporal dos encalhes série mensal

In [ ]:
# Agrupa encalhes por mês e conta ocorrências
serie_mensal = df_temporal.resample('ME').size().reset_index()
serie_mensal.columns = ['Mês', 'Encalhes']

fig, ax = plt.subplots(figsize=(14, 5))
sns.lineplot(data=serie_mensal, x='Mês', y='Encalhes', ax=ax, color='steelblue', linewidth=1.5)
ax.set_title('Evolução mensal dos encalhes de fauna marinha — Litoral de SP', fontsize=20, pad=12)
ax.set_xlabel('Data')
ax.set_ylabel('Número de encalhes')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('\nMeses com mais encalhes:')
print(serie_mensal.nlargest(5, 'Encalhes').to_string(index=False))


1.3 Média de encalhes por mês do calendário (2019–2025)

In [ ]:
# Extrair mês e ano do índice datetime
df_temporal['mes_do_ano'] = df_temporal.index.month
df_temporal['ano']        = df_temporal.index.year

# Passo 1: conta encalhes por (ano, mês)
contagem_por_mes_ano = (
    df_temporal
    .groupby(['ano', 'mes_do_ano'])
    .size()
    .reset_index(name='Encalhes')
)

# Passo 2: tira a média de cada mês do calendário ao longo dos anos
sazonalidade = (
    contagem_por_mes_ano
    .groupby('mes_do_ano')['Encalhes']
    .mean()
    .reset_index()
)
sazonalidade.columns = ['Mês', 'Média de encalhes']

# Nomes abreviados dos meses para o eixo X
nomes_meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
               'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
sazonalidade['Nome'] = sazonalidade['Mês'].apply(lambda m: nomes_meses[m - 1])

fig, ax = plt.subplots(figsize=(12, 5))

cores_saz = ['#e08080' if m in [8, 9] else '#5b9bd5' for m in sazonalidade['Mês']]

bars_saz = ax.bar(
    sazonalidade['Nome'],
    sazonalidade['Média de encalhes'],
    color=cores_saz,
    edgecolor='white',
    linewidth=0.8,
    width=0.6,
)

# Rótulo no topo de cada barra
for bar in bars_saz:
    h = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        h + sazonalidade['Média de encalhes'].max() * 0.01,
        f'{h:.0f}',
        ha='center', va='bottom', fontsize=8.5, fontweight='bold'
    )

ax.set_xlabel('Mês')
ax.set_ylabel('Média de encalhes por ano')
ax.set_ylim(0, sazonalidade['Média de encalhes'].max() * 1.18)
ax.spines[['top', 'right']].set_visible(False)
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
ax.tick_params(axis='x', labelsize=10)

# Legenda explicativa
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e08080', label='Meses de pico (ago/set)'),
    Patch(facecolor='#5b9bd5', label='Demais meses'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=10, framealpha=0.8)

plt.tight_layout()
plt.show()

print(f"\nMês com maior média: {sazonalidade.loc[sazonalidade['Média de encalhes'].idxmax(), 'Nome']}")
print(sazonalidade.sort_values('Média de encalhes', ascending=False)
      .assign(**{'Média de encalhes': lambda df: df['Média de encalhes'].round(1)})
      [['Nome', 'Média de encalhes']].to_string(index=False))


1.4 Encalhes por classe de Animal e 1.5 Proporção de animais vivos vs mortos

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({'figure.dpi': 70, 'savefig.dpi': 70})

col_classe = 'Espécies - Classe_ocor' if 'Espécies - Classe_ocor' in df_master.columns else 'Espécies - Classe'

contagem_classe = df_master[col_classe].value_counts().reset_index()
contagem_classe.columns = ['Classe', 'Encalhes']

condicao = df_master['Condição'].value_counts().reset_index()
condicao.columns = ['Condição', 'Contagem']

cores_classe = {'Aves': '#e41a1c', 'Reptilia': '#1D9E75', 'Mammalia': '#2171b5', 'Elasmobranchii': '#ff7f00'}
cores_condicao = {'Morto': '#d73027', 'Vivo': '#1a9850'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))  # era (14, 5)

cores_lista = [cores_classe.get(c, '#aaaaaa') for c in contagem_classe['Classe']]
bars1 = ax1.bar(contagem_classe['Classe'], contagem_classe['Encalhes'], color=cores_lista, edgecolor='white', linewidth=0.8, width=0.55)

for bar in bars1:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2, h + contagem_classe['Encalhes'].max() * 0.01, f'{int(h):,}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax1.set_title('Encalhes por classe de animal', fontsize=11, pad=8)
ax1.set_xlabel('Classe')
ax1.set_ylabel('Número de encalhes')
ax1.set_ylim(0, contagem_classe['Encalhes'].max() * 1.15)
ax1.spines[['top', 'right']].set_visible(False)
ax1.tick_params(axis='x', labelsize=9)
ax1.yaxis.grid(True, linestyle='--', alpha=0.4)
ax1.set_axisbelow(True)

cores_lista2 = [cores_condicao.get(c, '#aaaaaa') for c in condicao['Condição']]
bars2 = ax2.bar(condicao['Condição'], condicao['Contagem'], color=cores_lista2, edgecolor='white', linewidth=0.8, width=0.45)

total = condicao['Contagem'].sum()
for bar, (_, row) in zip(bars2, condicao.iterrows()):
    h = bar.get_height()
    pct = h / total * 100
    ax2.text(bar.get_x() + bar.get_width() / 2, h + total * 0.005, f'{int(h):,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax2.set_title('Condição dos animais encalhados', fontsize=11, pad=8)
ax2.set_xlabel('Condição')
ax2.set_ylabel('Número de registros')
ax2.set_ylim(0, condicao['Contagem'].max() * 1.20)
ax2.spines[['top', 'right']].set_visible(False)
ax2.tick_params(axis='x', labelsize=10)
ax2.yaxis.grid(True, linestyle='--', alpha=0.4)
ax2.set_axisbelow(True)

plt.tight_layout()
plt.show()

1.6 Top 15 praias — taxa de encalhes por esforço de monitoramento

In [ ]:
# Conta encalhes e esforço por praia e calcula a taxa
encalhes_por_praia = df_master['Praia'].value_counts().reset_index()
encalhes_por_praia.columns = ['Praia', 'Encalhes']

esforco_por_praia = esforco_monitora_filtrada['Praia'].value_counts().reset_index()
esforco_por_praia.columns = ['Praia', 'Esforço']

taxa_normalizada = pd.merge(encalhes_por_praia, esforco_por_praia, on='Praia', how='inner')
taxa_normalizada['Taxa'] = taxa_normalizada['Encalhes'] / taxa_normalizada['Esforço']
taxa_normalizada = taxa_normalizada.sort_values('Taxa', ascending=False)

top15 = taxa_normalizada.head(15).copy()
top15['Praia'] = top15['Praia'].apply(lambda x: x[:30] + '...' if len(x) > 30 else x)

norm = plt.Normalize(top15['Taxa'].min(), top15['Taxa'].max())
cmap = plt.get_cmap('YlOrRd')
fig, ax = plt.subplots(figsize=(14, 7))

bars = ax.barh(top15['Praia'], top15['Taxa'], color=[cmap(norm(v)) for v in top15['Taxa']])

for bar, val in zip(bars, top15['Taxa']):
    ax.text(bar.get_width() + top15['Taxa'].max() * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f}', va='center', fontsize=9, color='#444')

ax.set_xlabel('Encalhes por registro de esforço')
ax.set_ylabel('')
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

> **Resumo da preparação da Análise Exploratória e Séries Temporais:**  
> - **1.2** → Série temporal mensal: identifica tendências e picos ao longo dos anos  
> - **1.3** → Sazonalidade: revela quais meses concentram mais encalhes  
> - **1.4** → Encalhes por classe: mostra quais grupos taxonômicos são mais afetados  
> - **1.5** → Condição dos animais: proporção vivos vs mortos (objetivo específico do TCC)  
> - **1.6** → Taxa normalizada: compara praias com justiça, descontando o esforço de monitoramento  
>
> Os resultados desta fase respondem diretamente aos objetivos específicos do TCC e alimentam a Fase 4 (mapas espaciais).

---

## Geoprocessamento e Análise Espacial

1.1 Correção das coordenadas — Ocorrências

In [ ]:
def corrigir_coordenada(valor):
    s = str(valor).strip()
    if s in ('nan', 'None', ''):
        return np.nan

    # Remove todos os pontos e vírgulas
    sinal = '-' if s.startswith('-') else ''
    digitos = s.lstrip('-').replace('.', '').replace(',', '')

    # Sempre 2 dígitos na parte inteira para o litoral de SP
    resultado = sinal + digitos[:2] + '.' + digitos[2:]

    try:
        return float(resultado)
    except:
        return np.nan

ocorrencias_filtradas['Ponto - Lat'] = ocorrencias_filtradas['Ponto - Lat'].apply(corrigir_coordenada)
ocorrencias_filtradas['Ponto - Long'] = ocorrencias_filtradas['Ponto - Long'].apply(corrigir_coordenada)

print('Coordenadas corrigidas — primeiros valores:')
print(ocorrencias_filtradas[['Ponto - Lat', 'Ponto - Long']].head())


1.2 Correção das coordenadas — Esforços de Monitoramento

In [ ]:
# Aplicar a mesma função nas 4 colunas de coordenadas dos Esforços
colunas_coord = [
    'Latitude ponto inicial', 'Longitude ponto inicial',
    'Latitude ponto final',   'Longitude ponto final'
]

for col in colunas_coord:
    esforco_monitora_filtrada[col] = esforco_monitora_filtrada[col].apply(corrigir_coordenada)

print('Coordenadas dos Esforços corrigidas — primeiros valores:')
print(esforco_monitora_filtrada[colunas_coord].head())

1.3 Criando GeoDataFrame de pontos — Encalhes

In [ ]:
# Cria uma cópia do df_master com as coordenadas já corrigidas
df_geo = df_master.copy()
df_geo['Ponto - Lat'] = ocorrencias_filtradas['Ponto - Lat']
df_geo['Ponto - Long'] = ocorrencias_filtradas['Ponto - Long']

# Remove linhas sem coordenadas válidas
df_geo = df_geo.dropna(subset=['Ponto - Lat', 'Ponto - Long'])

# Cria coluna geometry: cada encalhe vira um ponto (longitude, latitude)
df_geo['geometry'] = df_geo.apply(
    lambda row: Point(row['Ponto - Long'], row['Ponto - Lat']), axis=1
)

# Transforma em GeoDataFrame com CRS WGS84 (padrão GPS / SIMBA)
gdf_encalhes = gpd.GeoDataFrame(df_geo, geometry='geometry', crs='EPSG:4326')

print(f' GeoDataFrame de encalhes criado:')
print(f'   Registros com coordenadas válidas: {len(gdf_encalhes)}')
print(f'   CRS: {gdf_encalhes.crs}')
print(gdf_encalhes[['Ponto - Lat', 'Ponto - Long', 'geometry']].head())

1.4 Criando GeoDataFrame de linhas — Esforços de Monitoramento

In [ ]:
# Cria uma linha geográfica para cada saída de monitoramento
esforco_geo = esforco_monitora_filtrada.dropna(subset=colunas_coord).copy()

esforco_geo['geometry'] = esforco_geo.apply(
    lambda row: LineString([
        (row['Longitude ponto inicial'], row['Latitude ponto inicial']),
        (row['Longitude ponto final'], row['Latitude ponto final'])
    ]), axis=1
)

gdf_esforcos = gpd.GeoDataFrame(esforco_geo, geometry='geometry', crs='EPSG:4326')

print(f'Total de rotas de monitoramento com coordenadas válidas: {len(gdf_esforcos)}')

1.5 Distribuição espacial dos encalhes\nLitoral do Estado de São Paulo

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({'figure.dpi': 70, 'savefig.dpi': 70})

url_sp = 'https://github.com/alura-cursos/curso_geopandas/raw/main/dados/Estado_SP.shp'
sp = gpd.read_file(url_sp)
sp = sp.to_crs(epsg=4326)

gdf_plot_geo = gdf_encalhes[
    gdf_encalhes['Ponto - Lat'].between(-26, -22) &
    gdf_encalhes['Ponto - Long'].between(-49, -44)
].copy()

col_classe = 'Espécies - Classe_ocor' if 'Espécies - Classe_ocor' in gdf_plot_geo.columns else 'Espécies - Classe'
cores_por_classe = {
    'Aves':           '#e41a1c',
    'Reptilia':       '#4daf4a',
    'Mammalia':       '#377eb8',
    'Elasmobranchii': '#ff7f00',
    'Sirenia':        '#984ea3',
}

fig, ax = plt.subplots(figsize=(6, 8))  # era (10, 13)

sp.plot(ax=ax, color='#f0f0f0', edgecolor='#cccccc', linewidth=0.5)

for classe in gdf_plot_geo[col_classe].dropna().unique():
    subset = gdf_plot_geo[gdf_plot_geo[col_classe] == classe]
    subset.plot(ax=ax, markersize=2, alpha=0.6, color=cores_por_classe.get(str(classe), '#aaaaaa'), label=str(classe))

ax.set_xlim(-49, -44.5)
ax.set_ylim(-25.4, -23.1)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(title='Classe', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

### 1.5b Distribuição espacial por município — Top 15


In [ ]:
# ── Bertioga — facet por classe ──────────────────────────────────────────

col_classe = 'Espécies - Classe_ocor' if 'Espécies - Classe_ocor' in gdf_encalhes.columns else 'Espécies - Classe'

classes_cfg = {
    'Aves':     {'cor': '#e41a1c', 'marker': 'o', 'size': 22, 'alpha': 0.80},
    'Reptilia': {'cor': '#1D9E75', 'marker': '^', 'size': 28, 'alpha': 0.85},
    'Mammalia': {'cor': '#2171b5', 'marker': 's', 'size': 24, 'alpha': 0.90},
}

mun_geo = sp[sp['NM_MUN'] == 'Bertioga'].copy()

if not mun_geo.empty:
    encalhes_filtrados = gdf_encalhes[
        gdf_encalhes['Ponto - Lat'].between(-26, -22) &
        gdf_encalhes['Ponto - Long'].between(-49, -44) &
        gdf_encalhes[col_classe].isin(['Aves', 'Reptilia', 'Mammalia'])
    ].copy()

    encalhes_dentro = gpd.sjoin(
        encalhes_filtrados,
        mun_geo[['NM_MUN', 'geometry']],
        how='inner', predicate='within'
    )

    bounds = mun_geo.total_bounds  # [xmin, ymin, xmax, ymax]
    w = bounds[2] - bounds[0]
    h = bounds[3] - bounds[1]

    # Margem fixa de 15% em cada lado
    mx = w * 0.15
    my = h * 0.15
    xmin, xmax = bounds[0] - mx, bounds[2] + mx
    ymin, ymax = bounds[1] - my, bounds[3] + my

    # Proporção real do município (graus lon × lat)
    # figsize: largura fixa 18, altura proporcional ao aspecto do bbox
    aspect = (ymax - ymin) / (xmax - xmin)
    fig_h = max(3.5, min(7.0, 18 * aspect / 3))  # por painel; clamp 3.5–7
    fig, axes = plt.subplots(1, 3, figsize=(18, fig_h))
    fig.suptitle('Bertioga  —  encalhes por classe', fontsize=16, fontweight='bold', y=1.03)

    for ax, (classe, cfg) in zip(axes, classes_cfg.items()):
        sp.plot(ax=ax, color='#f0f0f0', edgecolor='#bbbbbb', linewidth=0.3)
        mun_geo.plot(ax=ax, color='#fdd49e', edgecolor='#d94801', linewidth=1.2, zorder=2)

        subset = encalhes_dentro[encalhes_dentro[col_classe] == classe]

        if not subset.empty:
            ax.scatter(
                subset.geometry.x, subset.geometry.y,
                color=cfg['cor'], marker=cfg['marker'],
                s=cfg['size'], alpha=cfg['alpha'],
                edgecolors='white', linewidths=0.4, zorder=3,
            )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect('equal')   # mantém a forma geográfica correta
        ax.set_axis_off()
        n = len(subset) if not subset.empty else 0
        ax.set_title(f'{classe}  ({n})', fontsize=15, fontweight='bold',
                     color=cfg['cor'], pad=6)

    plt.tight_layout()
    plt.show()
else:
    print('Município não encontrado: Bertioga')

In [ ]:
# ── Ilha Comprida — facet por classe ──────────────────────────────────────────

col_classe = 'Espécies - Classe_ocor' if 'Espécies - Classe_ocor' in gdf_encalhes.columns else 'Espécies - Classe'

classes_cfg = {
    'Aves':     {'cor': '#e41a1c', 'marker': 'o', 'size': 22, 'alpha': 0.80},
    'Reptilia': {'cor': '#1D9E75', 'marker': '^', 'size': 28, 'alpha': 0.85},
    'Mammalia': {'cor': '#2171b5', 'marker': 's', 'size': 24, 'alpha': 0.90},
}

mun_geo = sp[sp['NM_MUN'] == 'Ilha Comprida'].copy()

if not mun_geo.empty:
    encalhes_filtrados = gdf_encalhes[
        gdf_encalhes['Ponto - Lat'].between(-26, -22) &
        gdf_encalhes['Ponto - Long'].between(-49, -44) &
        gdf_encalhes[col_classe].isin(['Aves', 'Reptilia', 'Mammalia'])
    ].copy()

    encalhes_dentro = gpd.sjoin(
        encalhes_filtrados,
        mun_geo[['NM_MUN', 'geometry']],
        how='inner', predicate='within'
    )

    bounds = mun_geo.total_bounds  # [xmin, ymin, xmax, ymax]
    w = bounds[2] - bounds[0]
    h = bounds[3] - bounds[1]

    # Margem fixa de 15% em cada lado
    mx = w * 0.15
    my = h * 0.15
    xmin, xmax = bounds[0] - mx, bounds[2] + mx
    ymin, ymax = bounds[1] - my, bounds[3] + my

    # Proporção real do município (graus lon × lat)
    # figsize: largura fixa 18, altura proporcional ao aspecto do bbox
    aspect = (ymax - ymin) / (xmax - xmin)
    fig_h = max(3.5, min(7.0, 18 * aspect / 3))  # por painel; clamp 3.5–7
    fig, axes = plt.subplots(1, 3, figsize=(18, fig_h))
    fig.suptitle('Ilha Comprida  —  encalhes por classe', fontsize=16, fontweight='bold', y=1.03)

    for ax, (classe, cfg) in zip(axes, classes_cfg.items()):
        sp.plot(ax=ax, color='#f0f0f0', edgecolor='#bbbbbb', linewidth=0.3)
        mun_geo.plot(ax=ax, color='#fdd49e', edgecolor='#d94801', linewidth=1.2, zorder=2)

        subset = encalhes_dentro[encalhes_dentro[col_classe] == classe]

        if not subset.empty:
            ax.scatter(
                subset.geometry.x, subset.geometry.y,
                color=cfg['cor'], marker=cfg['marker'],
                s=cfg['size'], alpha=cfg['alpha'],
                edgecolors='white', linewidths=0.4, zorder=3,
            )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect('equal')   # mantém a forma geográfica correta
        ax.set_axis_off()
        n = len(subset) if not subset.empty else 0
        ax.set_title(f'{classe}  ({n})', fontsize=14, fontweight='bold',
                     color=cfg['cor'], pad=6)

    plt.tight_layout()
    plt.show()
else:
    print('Município não encontrado: Ilha Comprida')

In [ ]:
# ── Cananéia — facet por classe ──────────────────────────────────────────

col_classe = 'Espécies - Classe_ocor' if 'Espécies - Classe_ocor' in gdf_encalhes.columns else 'Espécies - Classe'

classes_cfg = {
    'Aves':     {'cor': '#e41a1c', 'marker': 'o', 'size': 22, 'alpha': 0.80},
    'Reptilia': {'cor': '#1D9E75', 'marker': '^', 'size': 28, 'alpha': 0.85},
    'Mammalia': {'cor': '#2171b5', 'marker': 's', 'size': 24, 'alpha': 0.90},
}

mun_geo = sp[sp['NM_MUN'] == 'Cananéia'].copy()

if not mun_geo.empty:
    encalhes_filtrados = gdf_encalhes[
        gdf_encalhes['Ponto - Lat'].between(-26, -22) &
        gdf_encalhes['Ponto - Long'].between(-49, -44) &
        gdf_encalhes[col_classe].isin(['Aves', 'Reptilia', 'Mammalia'])
    ].copy()

    encalhes_dentro = gpd.sjoin(
        encalhes_filtrados,
        mun_geo[['NM_MUN', 'geometry']],
        how='inner', predicate='within'
    )

    bounds = mun_geo.total_bounds  # [xmin, ymin, xmax, ymax]
    w = bounds[2] - bounds[0]
    h = bounds[3] - bounds[1]

    # Margem fixa de 15% em cada lado
    mx = w * 0.15
    my = h * 0.15
    xmin, xmax = bounds[0] - mx, bounds[2] + mx
    ymin, ymax = bounds[1] - my, bounds[3] + my

    # Proporção real do município (graus lon × lat)
    # figsize: largura fixa 18, altura proporcional ao aspecto do bbox
    aspect = (ymax - ymin) / (xmax - xmin)
    fig_h = max(3.5, min(7.0, 18 * aspect / 3))  # por painel; clamp 3.5–7
    fig, axes = plt.subplots(1, 3, figsize=(18, fig_h))
    fig.suptitle('Cananéia  —  encalhes por classe', fontsize=18, fontweight='bold', y=1.03)

    for ax, (classe, cfg) in zip(axes, classes_cfg.items()):
        sp.plot(ax=ax, color='#f0f0f0', edgecolor='#bbbbbb', linewidth=0.3)
        mun_geo.plot(ax=ax, color='#fdd49e', edgecolor='#d94801', linewidth=1.2, zorder=2)

        subset = encalhes_dentro[encalhes_dentro[col_classe] == classe]

        if not subset.empty:
            ax.scatter(
                subset.geometry.x, subset.geometry.y,
                color=cfg['cor'], marker=cfg['marker'],
                s=cfg['size'], alpha=cfg['alpha'],
                edgecolors='white', linewidths=0.4, zorder=3,
            )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect('equal')   # mantém a forma geográfica correta
        ax.set_axis_off()
        n = len(subset) if not subset.empty else 0
        ax.set_title(f'{classe}  ({n})', fontsize=14, fontweight='bold',
                     color=cfg['cor'], pad=6)

    plt.tight_layout()
    plt.show()
else:
    print('Município não encontrado: Cananéia')

In [ ]:
# ── Caraguatatuba — facet por classe ──────────────────────────────────────────

col_classe = 'Espécies - Classe_ocor' if 'Espécies - Classe_ocor' in gdf_encalhes.columns else 'Espécies - Classe'

classes_cfg = {
    'Aves':     {'cor': '#e41a1c', 'marker': 'o', 'size': 22, 'alpha': 0.80},
    'Reptilia': {'cor': '#1D9E75', 'marker': '^', 'size': 28, 'alpha': 0.85},
    'Mammalia': {'cor': '#2171b5', 'marker': 's', 'size': 24, 'alpha': 0.90},
}

mun_geo = sp[sp['NM_MUN'] == 'Caraguatatuba'].copy()

if not mun_geo.empty:
    encalhes_filtrados = gdf_encalhes[
        gdf_encalhes['Ponto - Lat'].between(-26, -22) &
        gdf_encalhes['Ponto - Long'].between(-49, -44) &
        gdf_encalhes[col_classe].isin(['Aves', 'Reptilia', 'Mammalia'])
    ].copy()

    encalhes_dentro = gpd.sjoin(
        encalhes_filtrados,
        mun_geo[['NM_MUN', 'geometry']],
        how='inner', predicate='within'
    )

    bounds = mun_geo.total_bounds  # [xmin, ymin, xmax, ymax]
    w = bounds[2] - bounds[0]
    h = bounds[3] - bounds[1]

    # Margem fixa de 15% em cada lado
    mx = w * 0.15
    my = h * 0.15
    xmin, xmax = bounds[0] - mx, bounds[2] + mx
    ymin, ymax = bounds[1] - my, bounds[3] + my

    # Proporção real do município (graus lon × lat)
    # figsize: largura fixa 18, altura proporcional ao aspecto do bbox
    aspect = (ymax - ymin) / (xmax - xmin)
    fig_h = max(3.5, min(7.0, 18 * aspect / 3))  # por painel; clamp 3.5–7
    fig, axes = plt.subplots(1, 3, figsize=(18, fig_h))
    fig.suptitle('Caraguatatuba  —  encalhes por classe', fontsize=16, fontweight='bold', y=1.03)

    for ax, (classe, cfg) in zip(axes, classes_cfg.items()):
        sp.plot(ax=ax, color='#f0f0f0', edgecolor='#bbbbbb', linewidth=0.3)
        mun_geo.plot(ax=ax, color='#fdd49e', edgecolor='#d94801', linewidth=1.2, zorder=2)

        subset = encalhes_dentro[encalhes_dentro[col_classe] == classe]

        if not subset.empty:
            ax.scatter(
                subset.geometry.x, subset.geometry.y,
                color=cfg['cor'], marker=cfg['marker'],
                s=cfg['size'], alpha=cfg['alpha'],
                edgecolors='white', linewidths=0.4, zorder=3,
            )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect('equal')   # mantém a forma geográfica correta
        ax.set_axis_off()
        n = len(subset) if not subset.empty else 0
        ax.set_title(f'{classe}  ({n})', fontsize=14, fontweight='bold',
                     color=cfg['cor'], pad=6)

    plt.tight_layout()
    plt.show()
else:
    print('Município não encontrado: Caraguatatuba')

In [ ]:
# ── Guarujá — facet por classe ──────────────────────────────────────────

col_classe = 'Espécies - Classe_ocor' if 'Espécies - Classe_ocor' in gdf_encalhes.columns else 'Espécies - Classe'

classes_cfg = {
    'Aves':     {'cor': '#e41a1c', 'marker': 'o', 'size': 22, 'alpha': 0.80},
    'Reptilia': {'cor': '#1D9E75', 'marker': '^', 'size': 28, 'alpha': 0.85},
    'Mammalia': {'cor': '#2171b5', 'marker': 's', 'size': 24, 'alpha': 0.90},
}

mun_geo = sp[sp['NM_MUN'] == 'Guarujá'].copy()

if not mun_geo.empty:
    encalhes_filtrados = gdf_encalhes[
        gdf_encalhes['Ponto - Lat'].between(-26, -22) &
        gdf_encalhes['Ponto - Long'].between(-49, -44) &
        gdf_encalhes[col_classe].isin(['Aves', 'Reptilia', 'Mammalia'])
    ].copy()

    encalhes_dentro = gpd.sjoin(
        encalhes_filtrados,
        mun_geo[['NM_MUN', 'geometry']],
        how='inner', predicate='within'
    )

    bounds = mun_geo.total_bounds  # [xmin, ymin, xmax, ymax]
    w = bounds[2] - bounds[0]
    h = bounds[3] - bounds[1]

    # Margem fixa de 15% em cada lado
    mx = w * 0.15
    my = h * 0.15
    xmin, xmax = bounds[0] - mx, bounds[2] + mx
    ymin, ymax = bounds[1] - my, bounds[3] + my

    # Proporção real do município (graus lon × lat)
    # figsize: largura fixa 18, altura proporcional ao aspecto do bbox
    aspect = (ymax - ymin) / (xmax - xmin)
    fig_h = max(3.5, min(7.0, 18 * aspect / 3))  # por painel; clamp 3.5–7
    fig, axes = plt.subplots(1, 3, figsize=(18, fig_h))
    fig.suptitle('Guarujá  —  encalhes por classe', fontsize=16, fontweight='bold', y=1.03)

    for ax, (classe, cfg) in zip(axes, classes_cfg.items()):
        sp.plot(ax=ax, color='#f0f0f0', edgecolor='#bbbbbb', linewidth=0.3)
        mun_geo.plot(ax=ax, color='#fdd49e', edgecolor='#d94801', linewidth=1.2, zorder=2)

        subset = encalhes_dentro[encalhes_dentro[col_classe] == classe]

        if not subset.empty:
            ax.scatter(
                subset.geometry.x, subset.geometry.y,
                color=cfg['cor'], marker=cfg['marker'],
                s=cfg['size'], alpha=cfg['alpha'],
                edgecolors='white', linewidths=0.4, zorder=3,
            )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect('equal')   # mantém a forma geográfica correta
        ax.set_axis_off()
        n = len(subset) if not subset.empty else 0
        ax.set_title(f'{classe}  ({n})', fontsize=14, fontweight='bold',
                     color=cfg['cor'], pad=6)

    plt.tight_layout()
    plt.show()
else:
    print('Município não encontrado: Guarujá')

In [ ]:
# ── Iguape — facet por classe ──────────────────────────────────────────

col_classe = 'Espécies - Classe_ocor' if 'Espécies - Classe_ocor' in gdf_encalhes.columns else 'Espécies - Classe'

classes_cfg = {
    'Aves':     {'cor': '#e41a1c', 'marker': 'o', 'size': 22, 'alpha': 0.80},
    'Reptilia': {'cor': '#1D9E75', 'marker': '^', 'size': 28, 'alpha': 0.85},
    'Mammalia': {'cor': '#2171b5', 'marker': 's', 'size': 24, 'alpha': 0.90},
}

mun_geo = sp[sp['NM_MUN'] == 'Iguape'].copy()

if not mun_geo.empty:
    encalhes_filtrados = gdf_encalhes[
        gdf_encalhes['Ponto - Lat'].between(-26, -22) &
        gdf_encalhes['Ponto - Long'].between(-49, -44) &
        gdf_encalhes[col_classe].isin(['Aves', 'Reptilia', 'Mammalia'])
    ].copy()

    encalhes_dentro = gpd.sjoin(
        encalhes_filtrados,
        mun_geo[['NM_MUN', 'geometry']],
        how='inner', predicate='within'
    )

    bounds = mun_geo.total_bounds  # [xmin, ymin, xmax, ymax]
    w = bounds[2] - bounds[0]
    h = bounds[3] - bounds[1]

    # Margem fixa de 15% em cada lado
    mx = w * 0.15
    my = h * 0.15
    xmin, xmax = bounds[0] - mx, bounds[2] + mx
    ymin, ymax = bounds[1] - my, bounds[3] + my

    # Proporção real do município (graus lon × lat)
    # figsize: largura fixa 18, altura proporcional ao aspecto do bbox
    aspect = (ymax - ymin) / (xmax - xmin)
    fig_h = max(3.5, min(7.0, 18 * aspect / 3))  # por painel; clamp 3.5–7
    fig, axes = plt.subplots(1, 3, figsize=(18, fig_h))
    fig.suptitle('Iguape  —  encalhes por classe', fontsize=16, fontweight='bold', y=1.03)

    for ax, (classe, cfg) in zip(axes, classes_cfg.items()):
        sp.plot(ax=ax, color='#f0f0f0', edgecolor='#bbbbbb', linewidth=0.3)
        mun_geo.plot(ax=ax, color='#fdd49e', edgecolor='#d94801', linewidth=1.2, zorder=2)

        subset = encalhes_dentro[encalhes_dentro[col_classe] == classe]

        if not subset.empty:
            ax.scatter(
                subset.geometry.x, subset.geometry.y,
                color=cfg['cor'], marker=cfg['marker'],
                s=cfg['size'], alpha=cfg['alpha'],
                edgecolors='white', linewidths=0.4, zorder=3,
            )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect('equal')   # mantém a forma geográfica correta
        ax.set_axis_off()
        n = len(subset) if not subset.empty else 0
        ax.set_title(f'{classe}  ({n})', fontsize=14, fontweight='bold',
                     color=cfg['cor'], pad=6)

    plt.tight_layout()
    plt.show()
else:
    print('Município não encontrado: Iguape')

In [ ]:
# ── Ilhabela — facet por classe ──────────────────────────────────────────

col_classe = 'Espécies - Classe_ocor' if 'Espécies - Classe_ocor' in gdf_encalhes.columns else 'Espécies - Classe'

classes_cfg = {
    'Aves':     {'cor': '#e41a1c', 'marker': 'o', 'size': 22, 'alpha': 0.80},
    'Reptilia': {'cor': '#1D9E75', 'marker': '^', 'size': 28, 'alpha': 0.85},
    'Mammalia': {'cor': '#2171b5', 'marker': 's', 'size': 24, 'alpha': 0.90},
}

mun_geo = sp[sp['NM_MUN'] == 'Ilhabela'].copy()

if not mun_geo.empty:
    encalhes_filtrados = gdf_encalhes[
        gdf_encalhes['Ponto - Lat'].between(-26, -22) &
        gdf_encalhes['Ponto - Long'].between(-49, -44) &
        gdf_encalhes[col_classe].isin(['Aves', 'Reptilia', 'Mammalia'])
    ].copy()

    encalhes_dentro = gpd.sjoin(
        encalhes_filtrados,
        mun_geo[['NM_MUN', 'geometry']],
        how='inner', predicate='within'
    )

    bounds = mun_geo.total_bounds  # [xmin, ymin, xmax, ymax]
    w = bounds[2] - bounds[0]
    h = bounds[3] - bounds[1]

    # Margem fixa de 15% em cada lado
    mx = w * 0.15
    my = h * 0.15
    xmin, xmax = bounds[0] - mx, bounds[2] + mx
    ymin, ymax = bounds[1] - my, bounds[3] + my

    # Proporção real do município (graus lon × lat)
    # figsize: largura fixa 18, altura proporcional ao aspecto do bbox
    aspect = (ymax - ymin) / (xmax - xmin)
    fig_h = max(3.5, min(7.0, 18 * aspect / 3))  # por painel; clamp 3.5–7
    fig, axes = plt.subplots(1, 3, figsize=(18, fig_h))
    fig.suptitle('Ilhabela  —  encalhes por classe', fontsize=16, fontweight='bold', y=1.03)

    for ax, (classe, cfg) in zip(axes, classes_cfg.items()):
        sp.plot(ax=ax, color='#f0f0f0', edgecolor='#bbbbbb', linewidth=0.3)
        mun_geo.plot(ax=ax, color='#fdd49e', edgecolor='#d94801', linewidth=1.2, zorder=2)

        subset = encalhes_dentro[encalhes_dentro[col_classe] == classe]

        if not subset.empty:
            ax.scatter(
                subset.geometry.x, subset.geometry.y,
                color=cfg['cor'], marker=cfg['marker'],
                s=cfg['size'], alpha=cfg['alpha'],
                edgecolors='white', linewidths=0.4, zorder=3,
            )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect('equal')   # mantém a forma geográfica correta
        ax.set_axis_off()
        n = len(subset) if not subset.empty else 0
        ax.set_title(f'{classe}  ({n})', fontsize=14, fontweight='bold',
                     color=cfg['cor'], pad=6)

    plt.tight_layout()
    plt.show()
else:
    print('Município não encontrado: Ilhabela')

In [ ]:
# ── Itanhaém — facet por classe ──────────────────────────────────────────

col_classe = 'Espécies - Classe_ocor' if 'Espécies - Classe_ocor' in gdf_encalhes.columns else 'Espécies - Classe'

classes_cfg = {
    'Aves':     {'cor': '#e41a1c', 'marker': 'o', 'size': 22, 'alpha': 0.80},
    'Reptilia': {'cor': '#1D9E75', 'marker': '^', 'size': 28, 'alpha': 0.85},
    'Mammalia': {'cor': '#2171b5', 'marker': 's', 'size': 24, 'alpha': 0.90},
}

mun_geo = sp[sp['NM_MUN'] == 'Itanhaém'].copy()

if not mun_geo.empty:
    encalhes_filtrados = gdf_encalhes[
        gdf_encalhes['Ponto - Lat'].between(-26, -22) &
        gdf_encalhes['Ponto - Long'].between(-49, -44) &
        gdf_encalhes[col_classe].isin(['Aves', 'Reptilia', 'Mammalia'])
    ].copy()

    encalhes_dentro = gpd.sjoin(
        encalhes_filtrados,
        mun_geo[['NM_MUN', 'geometry']],
        how='inner', predicate='within'
    )

    bounds = mun_geo.total_bounds  # [xmin, ymin, xmax, ymax]
    w = bounds[2] - bounds[0]
    h = bounds[3] - bounds[1]

    # Margem fixa de 15% em cada lado
    mx = w * 0.15
    my = h * 0.15
    xmin, xmax = bounds[0] - mx, bounds[2] + mx
    ymin, ymax = bounds[1] - my, bounds[3] + my

    # Proporção real do município (graus lon × lat)
    # figsize: largura fixa 18, altura proporcional ao aspecto do bbox
    aspect = (ymax - ymin) / (xmax - xmin)
    fig_h = max(3.5, min(7.0, 18 * aspect / 3))  # por painel; clamp 3.5–7
    fig, axes = plt.subplots(1, 3, figsize=(18, fig_h))
    fig.suptitle('Itanhaém  —  encalhes por classe', fontsize=16, fontweight='bold', y=1.03)

    for ax, (classe, cfg) in zip(axes, classes_cfg.items()):
        sp.plot(ax=ax, color='#f0f0f0', edgecolor='#bbbbbb', linewidth=0.3)
        mun_geo.plot(ax=ax, color='#fdd49e', edgecolor='#d94801', linewidth=1.2, zorder=2)

        subset = encalhes_dentro[encalhes_dentro[col_classe] == classe]

        if not subset.empty:
            ax.scatter(
                subset.geometry.x, subset.geometry.y,
                color=cfg['cor'], marker=cfg['marker'],
                s=cfg['size'], alpha=cfg['alpha'],
                edgecolors='white', linewidths=0.4, zorder=3,
            )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect('equal')   # mantém a forma geográfica correta
        ax.set_axis_off()
        n = len(subset) if not subset.empty else 0
        ax.set_title(f'{classe}  ({n})', fontsize=14, fontweight='bold',
                     color=cfg['cor'], pad=6)

    plt.tight_layout()
    plt.show()
else:
    print('Município não encontrado: Itanhaém')

In [ ]:
# ── Mongaguá — facet por classe ──────────────────────────────────────────

col_classe = 'Espécies - Classe_ocor' if 'Espécies - Classe_ocor' in gdf_encalhes.columns else 'Espécies - Classe'

classes_cfg = {
    'Aves':     {'cor': '#e41a1c', 'marker': 'o', 'size': 22, 'alpha': 0.80},
    'Reptilia': {'cor': '#1D9E75', 'marker': '^', 'size': 28, 'alpha': 0.85},
    'Mammalia': {'cor': '#2171b5', 'marker': 's', 'size': 24, 'alpha': 0.90},
}

mun_geo = sp[sp['NM_MUN'] == 'Mongaguá'].copy()

if not mun_geo.empty:
    encalhes_filtrados = gdf_encalhes[
        gdf_encalhes['Ponto - Lat'].between(-26, -22) &
        gdf_encalhes['Ponto - Long'].between(-49, -44) &
        gdf_encalhes[col_classe].isin(['Aves', 'Reptilia', 'Mammalia'])
    ].copy()

    encalhes_dentro = gpd.sjoin(
        encalhes_filtrados,
        mun_geo[['NM_MUN', 'geometry']],
        how='inner', predicate='within'
    )

    bounds = mun_geo.total_bounds  # [xmin, ymin, xmax, ymax]
    w = bounds[2] - bounds[0]
    h = bounds[3] - bounds[1]

    # Margem fixa de 15% em cada lado
    mx = w * 0.15
    my = h * 0.15
    xmin, xmax = bounds[0] - mx, bounds[2] + mx
    ymin, ymax = bounds[1] - my, bounds[3] + my

    # Proporção real do município (graus lon × lat)
    # figsize: largura fixa 18, altura proporcional ao aspecto do bbox
    aspect = (ymax - ymin) / (xmax - xmin)
    fig_h = max(3.5, min(7.0, 18 * aspect / 3))  # por painel; clamp 3.5–7
    fig, axes = plt.subplots(1, 3, figsize=(18, fig_h))
    fig.suptitle('Mongaguá  —  encalhes por classe', fontsize=16, fontweight='bold', y=1.03)

    for ax, (classe, cfg) in zip(axes, classes_cfg.items()):
        sp.plot(ax=ax, color='#f0f0f0', edgecolor='#bbbbbb', linewidth=0.3)
        mun_geo.plot(ax=ax, color='#fdd49e', edgecolor='#d94801', linewidth=1.2, zorder=2)

        subset = encalhes_dentro[encalhes_dentro[col_classe] == classe]

        if not subset.empty:
            ax.scatter(
                subset.geometry.x, subset.geometry.y,
                color=cfg['cor'], marker=cfg['marker'],
                s=cfg['size'], alpha=cfg['alpha'],
                edgecolors='white', linewidths=0.4, zorder=3,
            )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect('equal')   # mantém a forma geográfica correta
        ax.set_axis_off()
        n = len(subset) if not subset.empty else 0
        ax.set_title(f'{classe}  ({n})', fontsize=14, fontweight='bold',
                     color=cfg['cor'], pad=6)

    plt.tight_layout()
    plt.show()
else:
    print('Município não encontrado: Mongaguá')

In [ ]:
# ── Peruíbe — facet por classe ──────────────────────────────────────────

col_classe = 'Espécies - Classe_ocor' if 'Espécies - Classe_ocor' in gdf_encalhes.columns else 'Espécies - Classe'

classes_cfg = {
    'Aves':     {'cor': '#e41a1c', 'marker': 'o', 'size': 22, 'alpha': 0.80},
    'Reptilia': {'cor': '#1D9E75', 'marker': '^', 'size': 28, 'alpha': 0.85},
    'Mammalia': {'cor': '#2171b5', 'marker': 's', 'size': 24, 'alpha': 0.90},
}

mun_geo = sp[sp['NM_MUN'] == 'Peruíbe'].copy()

if not mun_geo.empty:
    encalhes_filtrados = gdf_encalhes[
        gdf_encalhes['Ponto - Lat'].between(-26, -22) &
        gdf_encalhes['Ponto - Long'].between(-49, -44) &
        gdf_encalhes[col_classe].isin(['Aves', 'Reptilia', 'Mammalia'])
    ].copy()

    encalhes_dentro = gpd.sjoin(
        encalhes_filtrados,
        mun_geo[['NM_MUN', 'geometry']],
        how='inner', predicate='within'
    )

    bounds = mun_geo.total_bounds  # [xmin, ymin, xmax, ymax]
    w = bounds[2] - bounds[0]
    h = bounds[3] - bounds[1]

    # Margem fixa de 15% em cada lado
    mx = w * 0.15
    my = h * 0.15
    xmin, xmax = bounds[0] - mx, bounds[2] + mx
    ymin, ymax = bounds[1] - my, bounds[3] + my

    # Proporção real do município (graus lon × lat)
    # figsize: largura fixa 18, altura proporcional ao aspecto do bbox
    aspect = (ymax - ymin) / (xmax - xmin)
    fig_h = max(3.5, min(7.0, 18 * aspect / 3))  # por painel; clamp 3.5–7
    fig, axes = plt.subplots(1, 3, figsize=(18, fig_h))
    fig.suptitle('Peruíbe  —  encalhes por classe', fontsize=16, fontweight='bold', y=1.03)

    for ax, (classe, cfg) in zip(axes, classes_cfg.items()):
        sp.plot(ax=ax, color='#f0f0f0', edgecolor='#bbbbbb', linewidth=0.3)
        mun_geo.plot(ax=ax, color='#fdd49e', edgecolor='#d94801', linewidth=1.2, zorder=2)

        subset = encalhes_dentro[encalhes_dentro[col_classe] == classe]

        if not subset.empty:
            ax.scatter(
                subset.geometry.x, subset.geometry.y,
                color=cfg['cor'], marker=cfg['marker'],
                s=cfg['size'], alpha=cfg['alpha'],
                edgecolors='white', linewidths=0.4, zorder=3,
            )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect('equal')   # mantém a forma geográfica correta
        ax.set_axis_off()
        n = len(subset) if not subset.empty else 0
        ax.set_title(f'{classe}  ({n})', fontsize=14, fontweight='bold',
                     color=cfg['cor'], pad=6)

    plt.tight_layout()
    plt.show()
else:
    print('Município não encontrado: Peruíbe')

In [ ]:
# ── Praia Grande — facet por classe ──────────────────────────────────────────

col_classe = 'Espécies - Classe_ocor' if 'Espécies - Classe_ocor' in gdf_encalhes.columns else 'Espécies - Classe'

classes_cfg = {
    'Aves':     {'cor': '#e41a1c', 'marker': 'o', 'size': 22, 'alpha': 0.80},
    'Reptilia': {'cor': '#1D9E75', 'marker': '^', 'size': 28, 'alpha': 0.85},
    'Mammalia': {'cor': '#2171b5', 'marker': 's', 'size': 24, 'alpha': 0.90},
}

mun_geo = sp[sp['NM_MUN'] == 'Praia Grande'].copy()

if not mun_geo.empty:
    encalhes_filtrados = gdf_encalhes[
        gdf_encalhes['Ponto - Lat'].between(-26, -22) &
        gdf_encalhes['Ponto - Long'].between(-49, -44) &
        gdf_encalhes[col_classe].isin(['Aves', 'Reptilia', 'Mammalia'])
    ].copy()

    encalhes_dentro = gpd.sjoin(
        encalhes_filtrados,
        mun_geo[['NM_MUN', 'geometry']],
        how='inner', predicate='within'
    )

    bounds = mun_geo.total_bounds  # [xmin, ymin, xmax, ymax]
    w = bounds[2] - bounds[0]
    h = bounds[3] - bounds[1]

    # Margem fixa de 15% em cada lado
    mx = w * 0.15
    my = h * 0.15
    xmin, xmax = bounds[0] - mx, bounds[2] + mx
    ymin, ymax = bounds[1] - my, bounds[3] + my

    # Proporção real do município (graus lon × lat)
    # figsize: largura fixa 18, altura proporcional ao aspecto do bbox
    aspect = (ymax - ymin) / (xmax - xmin)
    fig_h = max(3.5, min(7.0, 18 * aspect / 3))  # por painel; clamp 3.5–7
    fig, axes = plt.subplots(1, 3, figsize=(18, fig_h))
    fig.suptitle('Praia Grande  —  encalhes por classe', fontsize=16, fontweight='bold', y=1.03)

    for ax, (classe, cfg) in zip(axes, classes_cfg.items()):
        sp.plot(ax=ax, color='#f0f0f0', edgecolor='#bbbbbb', linewidth=0.3)
        mun_geo.plot(ax=ax, color='#fdd49e', edgecolor='#d94801', linewidth=1.2, zorder=2)

        subset = encalhes_dentro[encalhes_dentro[col_classe] == classe]

        if not subset.empty:
            ax.scatter(
                subset.geometry.x, subset.geometry.y,
                color=cfg['cor'], marker=cfg['marker'],
                s=cfg['size'], alpha=cfg['alpha'],
                edgecolors='white', linewidths=0.4, zorder=3,
            )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect('equal')   # mantém a forma geográfica correta
        ax.set_axis_off()
        n = len(subset) if not subset.empty else 0
        ax.set_title(f'{classe}  ({n})', fontsize=14, fontweight='bold',
                     color=cfg['cor'], pad=6)

    plt.tight_layout()
    plt.show()
else:
    print('Município não encontrado: Praia Grande')

In [ ]:
# ── Santos — facet por classe ──────────────────────────────────────────

col_classe = 'Espécies - Classe_ocor' if 'Espécies - Classe_ocor' in gdf_encalhes.columns else 'Espécies - Classe'

classes_cfg = {
    'Aves':     {'cor': '#e41a1c', 'marker': 'o', 'size': 22, 'alpha': 0.80},
    'Reptilia': {'cor': '#1D9E75', 'marker': '^', 'size': 28, 'alpha': 0.85},
    'Mammalia': {'cor': '#2171b5', 'marker': 's', 'size': 24, 'alpha': 0.90},
}

mun_geo = sp[sp['NM_MUN'] == 'Santos'].copy()

if not mun_geo.empty:
    encalhes_filtrados = gdf_encalhes[
        gdf_encalhes['Ponto - Lat'].between(-26, -22) &
        gdf_encalhes['Ponto - Long'].between(-49, -44) &
        gdf_encalhes[col_classe].isin(['Aves', 'Reptilia', 'Mammalia'])
    ].copy()

    encalhes_dentro = gpd.sjoin(
        encalhes_filtrados,
        mun_geo[['NM_MUN', 'geometry']],
        how='inner', predicate='within'
    )

    bounds = mun_geo.total_bounds  # [xmin, ymin, xmax, ymax]
    w = bounds[2] - bounds[0]
    h = bounds[3] - bounds[1]

    # Margem fixa de 15% em cada lado
    mx = w * 0.15
    my = h * 0.15
    xmin, xmax = bounds[0] - mx, bounds[2] + mx
    ymin, ymax = bounds[1] - my, bounds[3] + my

    # Proporção real do município (graus lon × lat)
    # figsize: largura fixa 18, altura proporcional ao aspecto do bbox
    aspect = (ymax - ymin) / (xmax - xmin)
    fig_h = max(3.5, min(7.0, 18 * aspect / 3))  # por painel; clamp 3.5–7
    fig, axes = plt.subplots(1, 3, figsize=(18, fig_h))
    fig.suptitle('Santos  —  encalhes por classe', fontsize=16, fontweight='bold', y=1.03)

    for ax, (classe, cfg) in zip(axes, classes_cfg.items()):
        sp.plot(ax=ax, color='#f0f0f0', edgecolor='#bbbbbb', linewidth=0.3)
        mun_geo.plot(ax=ax, color='#fdd49e', edgecolor='#d94801', linewidth=1.2, zorder=2)

        subset = encalhes_dentro[encalhes_dentro[col_classe] == classe]

        if not subset.empty:
            ax.scatter(
                subset.geometry.x, subset.geometry.y,
                color=cfg['cor'], marker=cfg['marker'],
                s=cfg['size'], alpha=cfg['alpha'],
                edgecolors='white', linewidths=0.4, zorder=3,
            )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect('equal')   # mantém a forma geográfica correta
        ax.set_axis_off()
        n = len(subset) if not subset.empty else 0
        ax.set_title(f'{classe}  ({n})', fontsize=14, fontweight='bold',
                     color=cfg['cor'], pad=6)

    plt.tight_layout()
    plt.show()
else:
    print('Município não encontrado: Santos')

In [ ]:
# ── São Sebastião — facet por classe ──────────────────────────────────────────

col_classe = 'Espécies - Classe_ocor' if 'Espécies - Classe_ocor' in gdf_encalhes.columns else 'Espécies - Classe'

classes_cfg = {
    'Aves':     {'cor': '#e41a1c', 'marker': 'o', 'size': 22, 'alpha': 0.80},
    'Reptilia': {'cor': '#1D9E75', 'marker': '^', 'size': 28, 'alpha': 0.85},
    'Mammalia': {'cor': '#2171b5', 'marker': 's', 'size': 24, 'alpha': 0.90},
}

mun_geo = sp[sp['NM_MUN'] == 'São Sebastião'].copy()

if not mun_geo.empty:
    encalhes_filtrados = gdf_encalhes[
        gdf_encalhes['Ponto - Lat'].between(-26, -22) &
        gdf_encalhes['Ponto - Long'].between(-49, -44) &
        gdf_encalhes[col_classe].isin(['Aves', 'Reptilia', 'Mammalia'])
    ].copy()

    encalhes_dentro = gpd.sjoin(
        encalhes_filtrados,
        mun_geo[['NM_MUN', 'geometry']],
        how='inner', predicate='within'
    )

    bounds = mun_geo.total_bounds  # [xmin, ymin, xmax, ymax]
    w = bounds[2] - bounds[0]
    h = bounds[3] - bounds[1]

    # Margem fixa de 15% em cada lado
    mx = w * 0.15
    my = h * 0.15
    xmin, xmax = bounds[0] - mx, bounds[2] + mx
    ymin, ymax = bounds[1] - my, bounds[3] + my

    # Proporção real do município (graus lon × lat)
    # figsize: largura fixa 18, altura proporcional ao aspecto do bbox
    aspect = (ymax - ymin) / (xmax - xmin)
    fig_h = max(3.5, min(7.0, 18 * aspect / 3))  # por painel; clamp 3.5–7
    fig, axes = plt.subplots(1, 3, figsize=(18, fig_h))
    fig.suptitle('São Sebastião  —  encalhes por classe', fontsize=16, fontweight='bold', y=1.03)

    for ax, (classe, cfg) in zip(axes, classes_cfg.items()):
        sp.plot(ax=ax, color='#f0f0f0', edgecolor='#bbbbbb', linewidth=0.3)
        mun_geo.plot(ax=ax, color='#fdd49e', edgecolor='#d94801', linewidth=1.2, zorder=2)

        subset = encalhes_dentro[encalhes_dentro[col_classe] == classe]

        if not subset.empty:
            ax.scatter(
                subset.geometry.x, subset.geometry.y,
                color=cfg['cor'], marker=cfg['marker'],
                s=cfg['size'], alpha=cfg['alpha'],
                edgecolors='white', linewidths=0.4, zorder=3,
            )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect('equal')   # mantém a forma geográfica correta
        ax.set_axis_off()
        n = len(subset) if not subset.empty else 0
        ax.set_title(f'{classe}  ({n})', fontsize=14, fontweight='bold',
                     color=cfg['cor'], pad=6)

    plt.tight_layout()
    plt.show()
else:
    print('Município não encontrado: São Sebastião')

In [ ]:
# ── São Vicente — facet por classe ──────────────────────────────────────────

col_classe = 'Espécies - Classe_ocor' if 'Espécies - Classe_ocor' in gdf_encalhes.columns else 'Espécies - Classe'

classes_cfg = {
    'Aves':     {'cor': '#e41a1c', 'marker': 'o', 'size': 22, 'alpha': 0.80},
    'Reptilia': {'cor': '#1D9E75', 'marker': '^', 'size': 28, 'alpha': 0.85},
    'Mammalia': {'cor': '#2171b5', 'marker': 's', 'size': 24, 'alpha': 0.90},
}

mun_geo = sp[sp['NM_MUN'] == 'São Vicente'].copy()

if not mun_geo.empty:
    encalhes_filtrados = gdf_encalhes[
        gdf_encalhes['Ponto - Lat'].between(-26, -22) &
        gdf_encalhes['Ponto - Long'].between(-49, -44) &
        gdf_encalhes[col_classe].isin(['Aves', 'Reptilia', 'Mammalia'])
    ].copy()

    encalhes_dentro = gpd.sjoin(
        encalhes_filtrados,
        mun_geo[['NM_MUN', 'geometry']],
        how='inner', predicate='within'
    )

    bounds = mun_geo.total_bounds  # [xmin, ymin, xmax, ymax]
    w = bounds[2] - bounds[0]
    h = bounds[3] - bounds[1]

    # Margem fixa de 15% em cada lado
    mx = w * 0.15
    my = h * 0.15
    xmin, xmax = bounds[0] - mx, bounds[2] + mx
    ymin, ymax = bounds[1] - my, bounds[3] + my

    # Proporção real do município (graus lon × lat)
    # figsize: largura fixa 18, altura proporcional ao aspecto do bbox
    aspect = (ymax - ymin) / (xmax - xmin)
    fig_h = max(3.5, min(7.0, 18 * aspect / 3))  # por painel; clamp 3.5–7
    fig, axes = plt.subplots(1, 3, figsize=(18, fig_h))
    fig.suptitle('São Vicente  —  encalhes por classe', fontsize=16, fontweight='bold', y=1.03)

    for ax, (classe, cfg) in zip(axes, classes_cfg.items()):
        sp.plot(ax=ax, color='#f0f0f0', edgecolor='#bbbbbb', linewidth=0.3)
        mun_geo.plot(ax=ax, color='#fdd49e', edgecolor='#d94801', linewidth=1.2, zorder=2)

        subset = encalhes_dentro[encalhes_dentro[col_classe] == classe]

        if not subset.empty:
            ax.scatter(
                subset.geometry.x, subset.geometry.y,
                color=cfg['cor'], marker=cfg['marker'],
                s=cfg['size'], alpha=cfg['alpha'],
                edgecolors='white', linewidths=0.4, zorder=3,
            )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect('equal')   # mantém a forma geográfica correta
        ax.set_axis_off()
        n = len(subset) if not subset.empty else 0
        ax.set_title(f'{classe}  ({n})', fontsize=14, fontweight='bold',
                     color=cfg['cor'], pad=6)

    plt.tight_layout()
    plt.show()
else:
    print('Município não encontrado: São Vicente')

In [ ]:
# ── Ubatuba — facet por classe ──────────────────────────────────────────

col_classe = 'Espécies - Classe_ocor' if 'Espécies - Classe_ocor' in gdf_encalhes.columns else 'Espécies - Classe'

classes_cfg = {
    'Aves':     {'cor': '#e41a1c', 'marker': 'o', 'size': 22, 'alpha': 0.80},
    'Reptilia': {'cor': '#1D9E75', 'marker': '^', 'size': 28, 'alpha': 0.85},
    'Mammalia': {'cor': '#2171b5', 'marker': 's', 'size': 24, 'alpha': 0.90},
}

mun_geo = sp[sp['NM_MUN'] == 'Ubatuba'].copy()

if not mun_geo.empty:
    encalhes_filtrados = gdf_encalhes[
        gdf_encalhes['Ponto - Lat'].between(-26, -22) &
        gdf_encalhes['Ponto - Long'].between(-49, -44) &
        gdf_encalhes[col_classe].isin(['Aves', 'Reptilia', 'Mammalia'])
    ].copy()

    encalhes_dentro = gpd.sjoin(
        encalhes_filtrados,
        mun_geo[['NM_MUN', 'geometry']],
        how='inner', predicate='within'
    )

    bounds = mun_geo.total_bounds  # [xmin, ymin, xmax, ymax]
    w = bounds[2] - bounds[0]
    h = bounds[3] - bounds[1]

    # Margem fixa de 15% em cada lado
    mx = w * 0.15
    my = h * 0.15
    xmin, xmax = bounds[0] - mx, bounds[2] + mx
    ymin, ymax = bounds[1] - my, bounds[3] + my

    # Proporção real do município (graus lon × lat)
    # figsize: largura fixa 18, altura proporcional ao aspecto do bbox
    aspect = (ymax - ymin) / (xmax - xmin)
    fig_h = max(3.5, min(7.0, 18 * aspect / 3))  # por painel; clamp 3.5–7
    fig, axes = plt.subplots(1, 3, figsize=(18, fig_h))
    fig.suptitle('Ubatuba  —  encalhes por classe', fontsize=16, fontweight='bold', y=1.03)

    for ax, (classe, cfg) in zip(axes, classes_cfg.items()):
        sp.plot(ax=ax, color='#f0f0f0', edgecolor='#bbbbbb', linewidth=0.3)
        mun_geo.plot(ax=ax, color='#fdd49e', edgecolor='#d94801', linewidth=1.2, zorder=2)

        subset = encalhes_dentro[encalhes_dentro[col_classe] == classe]

        if not subset.empty:
            ax.scatter(
                subset.geometry.x, subset.geometry.y,
                color=cfg['cor'], marker=cfg['marker'],
                s=cfg['size'], alpha=cfg['alpha'],
                edgecolors='white', linewidths=0.4, zorder=3,
            )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect('equal')   # mantém a forma geográfica correta
        ax.set_axis_off()
        n = len(subset) if not subset.empty else 0
        ax.set_title(f'{classe}  ({n})', fontsize=14, fontweight='bold',
                     color=cfg['cor'], pad=6)

    plt.tight_layout()
    plt.show()
else:
    print('Município não encontrado: Ubatuba')

1.6 Mapa de calor — Densidade de encalhes por município\nLitoral do Estado de São Paulo

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))

# Merge entre shapefile e contagens de encalhe por município
# (ajuste 'NM_MUN' para o nome da coluna de município no seu shapefile)
contagem_mun = (
    gdf_encalhes['Cidade']
    .str.strip()
    .value_counts()
    .reset_index()
)
contagem_mun.columns = ['NM_MUN', 'Encalhes']

sp_heat = sp.merge(contagem_mun, on='NM_MUN', how='left')
sp_heat['Encalhes'] = sp_heat['Encalhes'].fillna(0)

# Base: todos os municípios de SP em cinza
sp.plot(ax=ax, color='#f5f5f5', edgecolor='#cccccc', linewidth=0.3)

# Colormap
cmap_heat = plt.cm.Reds

# Plot da camada de calor — sem criar colorbar ainda
sp_heat.plot(
    column='Encalhes',
    ax=ax,
    cmap=cmap_heat,
    linewidth=0.3,
    edgecolor='#cccccc',
    missing_kwds={'color': '#f5f5f5'},
    legend=False,      # ← desativa o colorbar padrão do geopandas
)

# ── Colorbar com make_axes_locatable (mesmo tamanho do mapa) ─────────────────
divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='4%', pad=0.12)

import matplotlib.cm as cm
import matplotlib.colors as mcolors

norm = mcolors.Normalize(
    vmin=sp_heat['Encalhes'].min(),
    vmax=sp_heat['Encalhes'].max()
)
sm = cm.ScalarMappable(cmap=cmap_heat, norm=norm)
sm.set_array([])

cb = plt.colorbar(sm, cax=cax)
cb.set_label('Número de encalhes', fontsize=10)

# Zoom na faixa litorânea
ax.set_xlim(-49.0, -44.4)
ax.set_ylim(-25.5, -22.8)
ax.set_xlabel('Longitude', fontsize=10)
ax.set_ylabel('Latitude', fontsize=10)

ax.tick_params(labelsize=9)

plt.tight_layout()
plt.show()


1.7 Top 15 municípios com mais encalhes registrados

In [ ]:
# Conta encalhes por município e plota os 15 com mais registros
encalhes_municipio = (
    gdf_encalhes['Cidade']
    .str.strip()                    # remove espaços extras
    .value_counts()
    .head(15)
    .reset_index()
)
encalhes_municipio.columns = ['Município', 'Encalhes']

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(
    encalhes_municipio['Município'][::-1],  # inverte para maior ficar no topo
    encalhes_municipio['Encalhes'][::-1],
    color='#d7301f'
)

# Rótulo com valor no final de cada barra
for bar, valor in zip(bars, encalhes_municipio['Encalhes'][::-1]):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
            f'{valor:,}', va='center', fontsize=9)


ax.set_xlabel('Número de encalhes')
ax.set_ylabel('')
ax.set_xlim(0, encalhes_municipio['Encalhes'].max() * 1.15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

# Mostra tabela também para confirmar os números
print(encalhes_municipio.to_string(index=False))

1.8 Hotspots de encalhes de fauna marinha\nLitoral do Estado de São Paulo — KDE

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({'figure.dpi': 60, 'savefig.dpi': 60})

lons = gdf_encalhes.geometry.x.values
lats = gdf_encalhes.geometry.y.values
mask = ~(np.isnan(lons) | np.isnan(lats))
lons, lats = lons[mask], lats[mask]

lon_min, lon_max = -49.0, -44.4
lat_min, lat_max = -25.3, -22.8

lon_grid = np.linspace(lon_min, lon_max, 150)  # era 200
lat_grid = np.linspace(lat_min, lat_max, 150)  # era 200
LON, LAT = np.meshgrid(lon_grid, lat_grid)

kde = gaussian_kde(np.vstack([lons, lats]), bw_method='scott')
Z = kde(np.vstack([LON.ravel(), LAT.ravel()])).reshape(LON.shape)
Z_norm = (Z - Z.min()) / (Z.max() - Z.min())

colors_kde = [(1,1,1,0),(1,0.95,0.6,0.3),(1,0.6,0.1,0.7),(0.85,0,0,0.95)]
cmap_kde = LinearSegmentedColormap.from_list('hotspot', colors_kde, N=256)

threshold = np.percentile(Z_norm, 70)
Z_plot = np.where(Z_norm >= threshold, Z_norm, np.nan)

fig, ax = plt.subplots(figsize=(8, 4.5))  # era (10, 5.5)

sp.plot(ax=ax, color='#f0f0f0', edgecolor='#bbbbbb', linewidth=0.4)
img = ax.imshow(Z_plot, extent=[lon_min, lon_max, lat_min, lat_max], origin='lower', cmap=cmap_kde, alpha=0.85, aspect='auto', vmin=threshold, vmax=1.0)

divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='4%', pad=0.12)
cb = plt.colorbar(img, cax=cax)
cb.set_label('Intensidade relativa de encalhes', fontsize=9)
cb.set_ticks([threshold, (threshold + 1) / 2, 1.0])
cb.set_ticklabels(['Baixa', 'Média', 'Alta'])

ax.set_xlim(lon_min, lon_max)
ax.set_ylim(lat_min, lat_max)
ax.set_xlabel('Longitude', fontsize=9)
ax.set_ylabel('Latitude', fontsize=9)
ax.tick_params(labelsize=8)
ax.grid(False)
plt.tight_layout()
plt.show()

> **Resumo do Geoprocessamento e Análise Espacial:**  
> - **1.1 / 1.2** → Coordenadas corrigidas para `float` nos dois datasets  
> - **1.3**/ **1.4** → `gdf_esforcos`: GeoDataFrame de linhas (cada saída de monitoramento = uma linha)  ** → `gdf_encalhes`: GeoDataFrame de pontos (cada encalhe = um ponto no mapa)  
> - **1.5** → Mapa estático com GeoPandas: distribuição por classe de animal  
> - **1.6** → Mapa de calor com Geopandas: hotspots de encalhes no litoral paulista  
> - **1.7** → Encalhes por município: ranking dos municípios mais afetados
> - **1.8** Hotspots de encalhes de fauna marinha\nLitoral do Estado de São Paulo — KDE
